# **Laboratorio 11: LLM y Agentes Autónomos 🤖**

MDS7202: Laboratorio de Programación Científica para Ciencia de Datos

### **Cuerpo Docente:**

- Profesores: Ignacio Meza, Sebastián Tinoco
- Auxiliar: Eduardo Moya
- Ayudantes: Nicolás Ojeda, Melanie Peña, Valentina Rojas

### **Equipo: SUPER IMPORTANTE - notebooks sin nombre no serán revisados**

- Nombre de alumno 1: Renato Pino
- Nombre de alumno 2: Valentina Abello

### **Link de repositorio de GitHub:** [Repositorio](https://github.com/Renato-98/MDS7202-1)

## **Temas a tratar**

- Reinforcement Learning
- Large Language Models

## **Reglas:**

- **Grupos de 2 personas**
- Cualquier duda fuera del horario de clases al foro. Mensajes al equipo docente serán respondidos por este medio.
- Prohibidas las copias.
- Pueden usar cualquer matrial del curso que estimen conveniente.

### **Objetivos principales del laboratorio**

- Resolución de problemas secuenciales usando Reinforcement Learning
- Habilitar un Chatbot para entregar respuestas útiles usando Large Language Models.

El laboratorio deberá ser desarrollado sin el uso indiscriminado de iteradores nativos de python (aka "for", "while"). La idea es que aprendan a exprimir al máximo las funciones optimizadas que nos entrega `pandas`, las cuales vale mencionar, son bastante más eficientes que los iteradores nativos sobre DataFrames.

## **1. Reinforcement Learning (2.0 puntos)**

En esta sección van a usar métodos de RL para resolver dos problemas interesantes: `Blackjack` y `LunarLander`.

In [ ]:
!pip install -qqq gymnasium stable_baselines3
!pip install -qqq swig
!pip install -qqq gymnasium[box2d]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 953.9/953.9 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 16.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.4/374.4 kB 6.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


### **1.1 Blackjack (1.0 puntos)**

<p align="center">
  <img src="https://www.recreoviral.com/wp-content/uploads/2016/08/s3.amazonaws.com-Math.gif"
" width="400">
</p>

La idea de esta subsección es que puedan implementar métodos de RL y así generar una estrategia para jugar el clásico juego Blackjack y de paso puedan ~~hacerse millonarios~~ aprender a resolver problemas mediante RL.

Comencemos primero preparando el ambiente. El siguiente bloque de código transforma las observaciones del ambiente a `np.array`:


In [ ]:
import gymnasium as gym
from gymnasium.spaces import MultiDiscrete
import numpy as np

class FlattenObservation(gym.ObservationWrapper):
    def __init__(self, env):
        super(FlattenObservation, self).__init__(env)
        self.observation_space = MultiDiscrete(np.array([32, 11, 2]))

    def observation(self, observation):
        return np.array(observation).flatten()

# Create and wrap the environment
env = gym.make("Blackjack-v1")
env = FlattenObservation(env)

#### **1.1.1 Descripción de MDP (0.2 puntos)**

Entregue una breve descripción sobre el ambiente [Blackjack](https://gymnasium.farama.org/environments/toy_text/blackjack/) y su formulación en MDP, distinguiendo de forma clara y concisa los estados, acciones y recompensas.

`escriba su respuesta acá`

El ambiente Blackjack simula el juego de cartas clásico en el que el objetivo es acercarse lo más posible a un puntaje de 21 sin superarlo. A continuación, describimos su formulación en el contexto de un Proceso de Decisión de Markov (MDP):

1. Estados (S):
Un estado describe la situación actual del juego y está compuesto por tres variables principales:

Suma de cartas del jugador: Un valor entre 0 y 31 (aunque en la práctica, los valores suelen estar entre 4 y 21).
Carta visible del dealer: Un valor entre 1 (As) y 10 (Rey, Reina, Jota se representan como 10).
Presencia de un As usable: True si el jugador tiene un As que puede contar como 11 sin exceder 21; False en caso contrario.
Ejemplo de un estado:

python
Copiar código
(15, 7, True)  # Suma del jugador: 15, carta del dealer: 7, el jugador tiene un As usable.
2. Acciones (A):
Las posibles acciones que puede tomar el jugador son:

Hit: Pedir una carta adicional.
Stick: Mantener el puntaje actual y finalizar el turno.
3. Recompensas (R):
La recompensa está directamente relacionada con el resultado del juego:

+1: El jugador gana.
0: Empate (push).
-1: El jugador pierde.
4. Transición (P):
La dinámica del juego depende de las reglas y las cartas:

Si el jugador elige "Hit", se le asigna una carta aleatoria, y su estado cambia dependiendo de la suma.
Si elige "Stick", el dealer jugará siguiendo sus reglas (debe pedir hasta llegar a 17 o más). El estado final y la recompensa dependen del resultado de esta jugada.
Blackjack como MDP:
Estados (S): Representan la situación del jugador y el dealer.
Acciones (A): Las decisiones del jugador (Hit o Stick).
Recompensas (R): Ganar, empatar o perder el juego.
Transiciones (P): Probabilidades de cambio de estado según las acciones y las reglas del juego.
Este modelo permite aplicar Reinforcement Learning para encontrar estrategias óptimas basadas en las recompensas acumuladas.

#### **1.1.2 Generando un Baseline (0.2 puntos)**

Simule un escenario en donde se escojan acciones aleatorias. Repita esta simulación 5000 veces y reporte el promedio y desviación de las recompensas. ¿Cómo calificaría el performance de esta política? ¿Cómo podría interpretar las recompensas obtenidas?

In [ ]:
# Inicializamos el ambiente
env = gym.make("Blackjack-v1")
env = FlattenObservation(env)

# Simulacion
num_episodes = 5000  # Numero de simulaciones
rewards = []  # Lista que almacena las recompensas de cada episodio

# Simulacion del juego aleatoria
for _ in range(num_episodes):
    done = False
    obs = env.reset()[0]  # Inicializamos el ambiente
    total_reward = 0

    while not done:
        action = env.action_space.sample()  # Acción aleatoria (Hit o Stick)
        obs, reward, done, _, _ = env.step(action)  # Ejecutamos la acción
        if done:
            total_reward += reward  # Acumulamos la recompensa al final del episodio

    rewards.append(total_reward)  # Guardamos la recompensa del episodio

# Calculamos el promedio y la desviacion estandar de las recompensas
average_reward = np.mean(rewards)
std_reward = np.std(rewards)

print(f"Promedio de recompensas: {average_reward}")
print(f"Desviacion estandar de recompensas: {std_reward}")

Promedio de recompensas: -0.4254
Desviacion estandar de recompensas: 0.882516198151626


El desempeño de la politica aleatoria es claramente suboptimo, como era de esperarse.
Un promedio de recompensas de -0.3748 indica que el jugador pierde mas partidas de las que gana. Esto sugiere que esta estrategia no utiliza informacion clave del estado para maximizar la probabilidad de ganar.

La alta desviacion alta(de 0.9 en este caso) muestra que las recompensas varian significativamente entre episodios. Esto ocurre porque, al depender de la aleatoriedad, hay partidas donde el jugador gana, pierde o empata en proporciones que fluctuan mucho.

#### **1.1.3 Entrenamiento de modelo (0.2 puntos)**

A partir del siguiente [enlace](https://stable-baselines3.readthedocs.io/en/master/guide/algos.html), escoja un modelo de `stable_baselines3` y entrenelo para resolver el ambiente `Blackjack`.

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env

# Inicializamos el ambiente
env = make_vec_env(lambda: FlattenObservation(gym.make("Blackjack-v1")), n_envs=1)

# Inicializamos el modelo PPO
model = PPO("MlpPolicy", env, verbose=1)

# Entrenamos el modelo
model.learn(total_timesteps=50000)

# Guardamos el modelo entrenado
model.save("ppo_blackjack_model")

Using cpu device
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 1.44     |
|    ep_rew_mean     | -0.4     |
| time/              |          |
|    fps             | 792      |
|    iterations      | 1        |
|    time_elapsed    | 2        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 1.3         |
|    ep_rew_mean          | -0.34       |
| time/                   |             |
|    fps                  | 614         |
|    iterations           | 2           |
|    time_elapsed         | 6           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.016167186 |
|    clip_fraction        | 0.299       |
|    clip_range           | 0.2         |
|    entropy_loss         | -0.677      |
|    explained_variance   | -0.0869     |
|    learning

#### **1.1.4 Evaluación de modelo (0.2 puntos)**

Repita el ejercicio 1.1.2 pero utilizando el modelo entrenado. ¿Cómo es el performance de su agente? ¿Es mejor o peor que el escenario baseline?

In [ ]:
# Cargamos el modelo entrenado
model = PPO.load("ppo_blackjack_model")

# Evaluamos el modelo
episodes = 1000
rewards = []

for _ in range(episodes):
    obs = env.reset()
    done = False
    total_reward = 0

    while not done:
        action, _ = model.predict(obs)  # El modelo predice la acción
        obs, reward, done, _= env.step(action)
        total_reward += reward  # Acumulamos la recompensa

    rewards.append(total_reward)

# Resultados
average_reward = np.mean(rewards)
std_reward = np.std(rewards)

print(f"Recompensa promedio después de entrenamiento: {average_reward}")
print(f"Desviación estándar: {std_reward}")

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


Recompensa promedio después de entrenamiento: -0.10199999809265137
Desviación estándar: 0.9620790481567383


El promedio de la recompensa es superior al del modelo baseline, pero sigue siendo negativa. Por lo tanto, este resultado muestra que el modelo aun no ha aprendido una politica optima o cercana a la optima.

Por otro tenemos una desviacion estandar muy grande, lo que indica que hay una gran variabilidad en las recompensas obtenidas durante la evaluacion, lo que sugiere que el modelo esta tomando decisiones inconsistentes o esta en un proceso de aprendizaje aun en las fases inicialees, ergo, el agente a veces gana mucho, pero otras veces pierde mucho.

Lo mas probable es que usando mas timestamps, el modelo llegue  a aprender una  una politica optima.

#### **1.1.5 Estudio de acciones (0.2 puntos)**

Genere una función que reciba un estado y retorne la accion del agente. Luego, use esta función para entregar la acción escogida frente a los siguientes escenarios:

- Suma de cartas del agente es 6, dealer muestra un 7, agente no tiene tiene un as
- Suma de cartas del agente es 19, dealer muestra un 3, agente tiene tiene un as

¿Son coherentes sus acciones con las reglas del juego?

Hint: ¿A que clase de python pertenecen los estados? Pruebe a usar el método `.reset` para saberlo.

In [ ]:
def get_action_for_state(state, model, env):
    """
    Esta función recibe el estado y devuelve la acción seleccionada por el agente.

    Args:
    - state: El estado del juego representado como un array de Numpy (total jugador, carta dealer, as utilizable).
    - model: El modelo entrenado de PPO.
    - env: El ambiente Gym con el que interactuar.

    Returns:
    - action: La acción tomada por el modelo (0 para 'Hit', 1 para 'Stick').
    """
    # Asegurarnos de que el estado esté en la forma correcta
    state = np.array(state).reshape(1, -1)  # Estado debe ser 2D para la predicción

    # Usamos el modelo para predecir la acción
    action, _ = model.predict(state)
    return action

# Escenario 1: Suma de cartas del agente es 6, dealer muestra un 7, agente no tiene as
state_1 = [6, 7, 0]  # [total jugador, carta visible del dealer, as utilizable]
action_1 = get_action_for_state(state_1, model, env)

# Escenario 2: Suma de cartas del agente es 19, dealer muestra un 3, agente tiene un as
state_2 = [19, 3, 1]  # [total jugador, carta visible del dealer, as utilizable]
action_2 = get_action_for_state(state_2, model, env)

# Imprimimos las acciones para ambos escenarios
print(f"Acción para el escenario 1 (Suma 6, Dealer muestra 7, sin As): {'Hit' if action_1 == 0 else 'Stick'}")
print(f"Acción para el escenario 2 (Suma 19, Dealer muestra 3, con As): {'Hit' if action_2 == 0 else 'Stick'}")


Acción para el escenario 1 (Suma 6, Dealer muestra 7, sin As): Stick
Acción para el escenario 2 (Suma 19, Dealer muestra 3, con As): Hit


Creo que no es muy coherente:

* Escenario 1: El agente debería haber "pedio una carta" (Hit), encuentro que este es un acierto ya que 6 es extremadamente bajo y es prácticamente imposible ganar si se elige quedarse ("Stick") y es imposible superar 21 si elige otra carta.

* Escenario 2: El agente debería haberse "quedado" (Stick), no haber "pedido carta" (Hit) ya que pedir aqui es demasiado arriesgado teniendo
un puntaje de 19 y un As usable, el jugador tiene una mano sólida y está cerca de 21.

### **1.2 LunarLander**

<p align="center">
  <img src="https://i.redd.it/097t6tk29zf51.jpg"
" width="400">
</p>

Similar a la sección 2.1, en esta sección usted se encargará de implementar una gente de RL que pueda resolver el ambiente `LunarLander`.

Comencemos preparando el ambiente:


In [ ]:
import gymnasium as gym
env = gym.make("LunarLander-v2", render_mode = "rgb_array", continuous = True) # notar el parámetro continuous = True

Noten que se especifica el parámetro `continuous = True`. ¿Que implicancias tiene esto sobre el ambiente?

Además, se le facilita la función `export_gif` para el ejercicio 2.2.4:

In [ ]:
import imageio
import numpy as np

def export_gif(model, n = 5):
  '''
  función que exporta a gif el comportamiento del agente en n episodios
  '''
  images = []
  for episode in range(n):
    obs = model.env.reset()
    img = model.env.render()
    done = False
    while not done:
      images.append(img)
      action, _ = model.predict(obs)
      obs, reward, done, info = model.env.step(action)
      img = model.env.render(mode="rgb_array")

  imageio.mimsave("agent_performance.gif", [np.array(img) for i, img in enumerate(images) if i%2 == 0], fps=29)

#### **1.2.1 Descripción de MDP (0.2 puntos)**

Entregue una breve descripción sobre el ambiente [LunarLander](https://gymnasium.farama.org/environments/box2d/lunar_lander/) y su formulación en MDP, distinguiendo de forma clara y concisa los estados, acciones y recompensas. ¿Como se distinguen las acciones de este ambiente en comparación a `Blackjack`?

Nota: recuerde que se especificó el parámetro `continuous = True`

`escriba su respuesta acá`

El ambiente LunarLander es un simulador en el que un agente controla un módulo lunar que debe aterrizar de manera segura sobre una superficie específica en un entorno de gravedad. El objetivo es minimizar el consumo de combustible mientras se logra un aterrizaje exitoso.

1. Estados (S):
Los estados en LunarLander son representados por un vector continuo de 8 valores:

Posición horizontal del módulo lunar (x).
Posición vertical del módulo lunar (y).
Velocidad horizontal (vx).
Velocidad vertical (vy).
Ángulo de orientación (θ).
Velocidad angular (ω).
Indicador del contacto de la pierna izquierda (binario: 0 o 1).
Indicador del contacto de la pierna derecha (binario: 0 o 1).
Estos valores juntos describen la situación completa del módulo en el ambiente.

2. Acciones (A):
Las acciones en LunarLander dependen de si el ambiente es continuo o discreto:

Ambiente Discreto:
Las acciones son enteros representando:

No hacer nada.
Encender el motor principal.
Encender el motor lateral izquierdo.
Encender el motor lateral derecho.

Ambiente Continuo:
Las acciones son vectores continuos que controlan la intensidad del motor principal y de los motores laterales.


Utilizar el parámetro continuous=True hace que el espacio de acción se convierta en continuo, lo que significa que las acciones están representadas por un vector de valores reales (en lugar de valores enteros). Según [Link](https://gymnasium.farama.org/environments/box2d/lunar_lander/#action-space), se pasa un vector que tiene dos componentes:
El primer valor controla la potencia del motor principal en rango continuo entre [-1,1] y el segundo valor controla la potencia de los motores laterales, también entre [-1,1].




#### **1.2.2 Generando un Baseline (0.2 puntos)**

Simule un escenario en donde se escojan acciones aleatorias. Repita esta simulación 10 veces y reporte el promedio y desviación de las recompensas. ¿Cómo calificaría el performance de esta política?

In [ ]:
import numpy as np
import gymnasium as gym

# Inicializar el entorno
env = gym.make("LunarLander-v2", render_mode="rgb_array", continuous=True)

# Lista para almacenar las recompensas totales por episodio
recompensas = []

# Simular 10 episodios
n_episodios = 10
for _ in range(n_episodios):
    obs, _ = env.reset()
    recompensa_total = 0
    terminado = False

    while not terminado:
        # Seleccionar una acción aleatoria
        accion = env.action_space.sample()
        obs, recompensa, terminado, truncado, _ = env.step(accion)
        recompensa_total += recompensa

    recompensas.append(recompensa_total)

# Calcular metricas
promedio_recompensas = np.mean(recompensas)
desviacion_recompensas = np.std(recompensas)

print(f"Promedio de recompensas: {promedio_recompensas}")
print(f"Desviacion estándar de recompensas: {desviacion_recompensas}")

Promedio de recompensas: -216.42919430407542
Desviacion estándar de recompensas: 139.18429194976537


Este baseline establece que una política aleatoria tiene un rendimiento deficiente con un promedio de recompensas cercano a -220.77, para este intento. Un agente entrenado con RL debería superar este baseline al aprender a tomar decisiones informadas que minimicen colisiones y maximicen las recompensas.De igual forma, la alta desviación estandar indica una gran variabilidad en las recompensas obtenidas entre episodios

#### **1.2.3 Entrenamiento de modelo (0.2 puntos)**

A partir del siguiente [enlace](https://stable-baselines3.readthedocs.io/en/master/guide/algos.html), escoja un modelo de `stable_baselines3` y entrenelo para resolver el ambiente `LunarLander` **usando 10000 timesteps de entrenamiento**.

In [ ]:
# Probaremos con TD3 que parece ser un modelo mas moderno.. y espero que por lo tanto, mejor jeje
import gymnasium as gym
from stable_baselines3 import TD3
from stable_baselines3.common.noise import NormalActionNoise
import numpy as np

# Crear el ambiente
env = gym.make("LunarLander-v2", render_mode="rgb_array", continuous=True)

# Configurar ruido para explorar acciones (TD3 requiere ruido en las acciones)
n_actions = env.action_space.shape[0]
action_noise = NormalActionNoise(mean=np.zeros(n_actions), sigma=0.1 * np.ones(n_actions))

# Crear el modelo TD3
model = TD3("MlpPolicy", env, action_noise=action_noise, verbose=1)

# Entrenar el modelo
model.learn(total_timesteps=10000)

# Guardar el modelo entrenado
model.save("td3_lunarlander")

# Cargar el modelo entrenado (si es necesario en el futuro)
model = TD3.load("td3_lunarlander")

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 79.2     |
|    ep_rew_mean     | -397     |
| time/              |          |
|    episodes        | 4        |
|    fps             | 46       |
|    time_elapsed    | 6        |
|    total_timesteps | 317      |
| train/             |          |
|    actor_loss      | 5.07     |
|    critic_loss     | 275      |
|    learning_rate   | 0.001    |
|    n_updates       | 216      |
---------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 103      |
|    ep_rew_mean     | -340     |
| time/              |          |
|    episodes        | 8        |
|    fps             | 38       |
|    time_elapsed    | 21       |
|    total_timesteps | 827      |
| train/             |          |
|    actor_loss      | 4.81     |
|    critic_loss     |

In [ ]:
import gymnasium as gym
from stable_baselines3 import PPO

# Crear el ambiente
env = gym.make("LunarLander-v2", render_mode="rgb_array", continuous=True)

# Crear el modelo PPO
model = PPO("MlpPolicy", env, verbose=1)

# Entrenar el modelo
model.learn(total_timesteps=10000)

# Guardar el modelo entrenado
model.save("ppo_lunarlander")

# Cargar y evaluar el modelo entrenado
model = PPO.load("ppo_lunarlander")

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 109      |
|    ep_rew_mean     | -245     |
| time/              |          |
|    fps             | 830      |
|    iterations      | 1        |
|    time_elapsed    | 2        |
|    total_timesteps | 2048     |
---------------------------------
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 110          |
|    ep_rew_mean          | -247         |
| time/                   |              |
|    fps                  | 613          |
|    iterations           | 2            |
|    time_elapsed         | 6            |
|    total_timesteps      | 4096         |
| train/                  |              |
|    approx_kl            | 0.0037077225 |
|    clip_fraction        | 0.0185       |
|    clip_range           | 0.2          |
|    en

#### **1.2.4 Evaluación de modelo (0.2 puntos)**

Repita el ejercicio 1.2.2 pero utilizando el modelo entrenado. ¿Cómo es el performance de su agente? ¿Es mejor o peor que el escenario baseline?

In [ ]:
# Evaluar el modelo en 10 episodios
recompensas = []
n_episodios = 10

for _ in range(n_episodios):
    obs, _ = env.reset()
    recompensa_total = 0
    terminado = False

    while not terminado:
        accion, _ = model.predict(obs, deterministic=True)
        obs, recompensa, terminado, truncado, _ = env.step(accion)
        recompensa_total += recompensa

    recompensas.append(recompensa_total)

promedio_recompensas = np.mean(recompensas)
desviacion_recompensas = np.std(recompensas)

print(f"Promedio de recompensas: {promedio_recompensas}")
print(f"Desviación estándar de recompensas: {desviacion_recompensas}")

Promedio de recompensas: -145.93911705057525
Desviación estándar de recompensas: 192.1806496849832


In [ ]:
import numpy as np

# Evaluar el modelo en 10 episodios
recompensas = []
n_episodios = 10

for _ in range(n_episodios):
    obs, _ = env.reset()
    recompensa_total = 0
    terminado = False

    while not terminado:
        accion, _ = model.predict(obs, deterministic=True)
        obs, recompensa, terminado, truncado, _ = env.step(accion)
        recompensa_total += recompensa

    recompensas.append(recompensa_total)

promedio_recompensas = np.mean(recompensas)
desviacion_recompensas = np.std(recompensas)

print(f"Promedio de recompensas: {promedio_recompensas}")
print(f"Desviación estándar de recompensas: {desviacion_recompensas}")

Promedio de recompensas: -99.69250235812589
Desviación estándar de recompensas: 70.21756163268714


El modelo entrenado muestra un ligero progreso respecto al baseline aleatorio almenos  en terminos de la desviacion estandar, sin embargo, aun presenta  gran variabilidad, lo que podria indicar que el agente no ha aprendido a resolver completamente el problema. Algunos episodios exitosos están elevando el promedio, mientras que otros episodios muestran comportamientos no optimos.

Por lo que los mas probable es que el modelo  necesite mas timesteps para alcanzar su potencial.



#### **1.2.5 Optimización de modelo (0.2 puntos)**

Repita los ejercicios 1.2.3 y 1.2.4 hasta obtener un nivel de recompensas promedio mayor a 50. Para esto, puede cambiar manualmente parámetros como:
- `total_timesteps`
- `learning_rate`
- `batch_size`

Una vez optimizado el modelo, use la función `export_gif` para estudiar el comportamiento de su agente en la resolución del ambiente y comente sobre sus resultados.

Adjunte el gif generado en su entrega (mejor aún si además adjuntan el gif en el markdown).

In [ ]:
from stable_baselines3 import TD3
from stable_baselines3.common.noise import NormalActionNoise

# Ajustes iniciales
n_actions = env.action_space.shape[0]
action_noise = NormalActionNoise(mean=np.zeros(n_actions), sigma=0.1 * np.ones(n_actions))

# Ajustes optimizados
model = TD3(
    "MlpPolicy",
    env,
    action_noise=action_noise,
    learning_rate=1e-4,  # Reducir tasa de aprendizaje
    batch_size=256,      # Aumentar batch size
    verbose=1
)

# Entrenamiento con más timesteps
model.learn(total_timesteps=100000)  # Aumentar timesteps

In [ ]:
# Evaluación del modelo optimizado
recompensas = []
n_episodios = 20

for _ in range(n_episodios):
    obs, _ = env.reset()
    recompensa_total = 0
    terminado = False

    while not terminado:
        accion, _ = model.predict(obs, deterministic=True)
        obs, recompensa, terminado, truncado, _ = env.step(accion)
        recompensa_total += recompensa

    recompensas.append(recompensa_total)

promedio_recompensas = np.mean(recompensas)
desviacion_recompensas = np.std(recompensas)

print(f"Promedio de recompensas después de optimización: {promedio_recompensas}")
print(f"Desviación estándar: {desviacion_recompensas}")


In [ ]:
# Generar GIF del comportamiento del agente
export_gif(model, n=5)

## **2. Large Language Models (4.0 puntos)**

En esta sección se enfocarán en habilitar un Chatbot que nos permita responder preguntas útiles a través de LLMs.

### **2.0 Configuración Inicial**

<p align="center">
  <img src="https://media1.tenor.com/m/uqAs9atZH58AAAAd/config-config-issue.gif"
" width="400">
</p>

Como siempre, cargamos todas nuestras API KEY al entorno:

In [ ]:
import getpass
import os

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google AI API key: ")

if "TAVILY_API_KEY" not in os.environ:
    os.environ["TAVILY_API_KEY"] = getpass.getpass("Enter your Tavily API key: ")

### **2.1 Retrieval Augmented Generation (1.5 puntos)**

<p align="center">
  <img src="https://y.yarn.co/218aaa02-c47e-4ec9-b1c9-07792a06a88f_text.gif"
" width="400">
</p>

El objetivo de esta subsección es que habiliten un chatbot que pueda responder preguntas usando información contenida en documentos PDF a través de **Retrieval Augmented Generation.**

#### **2.1.1 Reunir Documentos (0 puntos)**

Reuna documentos PDF sobre los que hacer preguntas siguiendo las siguientes instrucciones:
  - 2 documentos .pdf como mínimo.
  - 50 páginas de contenido como mínimo entre todos los documentos.
  - Ideas para documentos: Documentos relacionados a temas académicos, laborales o de ocio. Aprovechen este ejercicio para construir algo útil y/o relevante para ustedes!
  - Deben ocupar documentos reales, no pueden utilizar los mismos de la clase.
  - Deben registrar sus documentos en la siguiente [planilla](https://docs.google.com/spreadsheets/d/1Hy1w_dOiG2UCHJ8muyxhdKPZEPrrL7BNHm6E90imIIM/edit?usp=sharing). **NO PUEDEN USAR LOS MISMOS DOCUMENTOS QUE OTRO GRUPO**
  - **Recuerden adjuntar los documentos en su entrega**.

In [ ]:
%pip install --upgrade --quiet PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 3.7 MB/s eta 0:00:00


In [ ]:
# Si usted está utilizando Colabolatory le puede ser útil este código para cargar los archivos.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    path = '/content/drive/MyDrive/Lab_progra_DS/Lab_11/' # Ajustar al path propio
except:
    print('Ignorando conexión drive-colab')

Mounted at /content/drive


In [ ]:
import PyPDF2

doc_paths = [path + '/FAUNA-DE-CHILE.pdf', path + '/Biodiversidad.pdf'] # rellenar con los path a sus documentos

assert len(doc_paths) >= 2, "Deben adjuntar un mínimo de 2 documentos"

total_paginas = sum(len(PyPDF2.PdfReader(open(doc, "rb")).pages) for doc in doc_paths)
assert total_paginas >= 50, f"Páginas insuficientes: {total_paginas}"

#### **2.1.2 Vectorizar Documentos (0.2 puntos)**

Vectorice los documentos y almacene sus representaciones de manera acorde.

In [ ]:
!pip install -U langchain langchain-community faiss-cpu PyPDF2 openai

In [ ]:
!pip install langchain-google-genai # Install the required package

In [ ]:
from langchain.schema import Document
from langchain.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_google_genai.embeddings import GoogleGenerativeAIEmbeddings

# Paso 1: Leemos el texto de los documentos:
docs = []
for doc_path in doc_paths:
    with open(doc_path, "rb") as file:
        reader = PyPDF2.PdfReader(file)
        text = ""
        for page in reader.pages:
            text += page.extract_text()
        docs.append(Document(page_content=text, metadata={"source": doc_path}))

# Paso 2: Dividimos los documentos en fragmentos
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, # valores que recomendó colab jj
    chunk_overlap=200,
    separators=["\n\n", "\n", ".", "!", "?", "¿", "¡", ",", " ", ""],
)
splits = text_splitter.split_documents(docs)

In [ ]:
%pip install --upgrade --quiet  langchain-google-genai

In [ ]:
# Paso 3: Vectorizar los fragmentos
embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")
vectorstore = FAISS.from_documents(splits, embeddings)
vectorstore.save_local("vectorstore_faiss")

# Paso 4: Configuramos retriever desde vectorstore
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2}
)

#### **2.1.3 Habilitar RAG (0.3 puntos)**

Habilite la solución RAG a través de una *chain* y guárdela en una variable.

In [ ]:
from langchain.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain.chains import RetrievalQA

llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash", # modelo de lenguaje
    temperature=0, # probabilidad de "respuestas creativas"
    max_tokens=None, # sin tope de tokens
    timeout=None, # sin timeout
    max_retries=2, # número máximo de intentos
)

rag_template = '''
Eres un asistente experto en la fauna chilena.
Tu único rol es contestar preguntas del usuario a partir de información relevante que te sea proporcionada.
Responde siempre de la forma más completa posible y usando toda la información entregada.
Responde sólo lo que te pregunten a partir de la información relevante, NUNCA inventes una respuesta.

Información relevante: {context}
Pregunta: {question}
Respuesta útil:
'''

rag_prompt = PromptTemplate.from_template(rag_template)

retriever_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=False
)

rag_chain = (
    {
        "context": retriever_chain, # context lo obtendremos del retriever_chain
        "question": RunnablePassthrough(), # question pasará directo hacia el prompt
    }
    | rag_prompt # prompt con las variables question y context
    | llm # llm recibe el prompt y responde
    | StrOutputParser() # recuperamos sólo la respuesta
)

#### **2.1.4 Verificación de respuestas (0.5 puntos)**

Genere un listado de 3 tuplas ("pregunta", "respuesta correcta") y analice la respuesta de su solución para cada una. ¿Su solución RAG entrega las respuestas que esperaba?

Ejemplo de tupla:
- Pregunta: ¿Quién es el presidente de Chile?
- Respuesta correcta: El presidente de Chile es Gabriel Boric

In [ ]:
question = "¿Dónde habita la perdiz chilena?"
response = rag_chain.invoke(question)
print(response)

La perdiz chilena habita en campos de pastizales, arbustos bajos y campos agrícolas.



<p align="left">
  <img src="https://i.pinimg.com/736x/54/c3/27/54c32743e5c31e227b4414ca7aabee46.jpg"
" width="200">
</p>

#### **2.1.5 Sensibilidad de Hiperparámetros (0.5 puntos)**

Extienda el análisis del punto 2.1.4 analizando cómo cambian las respuestas entregadas cambiando los siguientes hiperparámetros:
- `Tamaño del chunk`. (*¿Cómo repercute que los chunks sean mas grandes o chicos?*)
- `La cantidad de chunks recuperados`. (*¿Qué pasa si se devuelven muchos/pocos chunks?*)
- `El tipo de búsqueda`. (*¿Cómo afecta el tipo de búsqueda a las respuestas de mi RAG?*)

- `Tamaño del chunk`.
  - Este afecta directamente la granulaidad de la información que se entrega al modelo. Chunks más pequeños podrían ser más precisos pero se pierde contexto, mientras que chunks entregan más contextopero pdorían incluir información irrelevante.
- `La cantidad de chunks recuperados`.
  - Valores bajos pueden limitar el contexto y entregar respuestas incompletas, mientras que valores muy altos podrían introducir ruido y generar respuestas irrelevantes.
- `El tipo de búsqueda`.
  - Este define cómo se mide la relevancia de los chinks recuperados, en este casi nosotros utilizamos `similarity` (similitud), que devuelve los chunks más cercanos a la consulta según los embeddings.

### **2.2 Agentes (1.0 puntos)**

<p align="center">
  <img src="https://media1.tenor.com/m/rcqnN2aJCSEAAAAd/secret-agent-man.gif"
" width="400">
</p>

Similar a la sección anterior, en esta sección se busca habilitar **Agentes** para obtener información a través de tools y así responder la pregunta del usuario.

#### **2.2.1 Tool de Tavily (0.2 puntos)**

Generar una *tool* que pueda hacer consultas al motor de búsqueda **Tavily**.

In [ ]:
from langchain import hub
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain.agents import create_react_agent, AgentExecutor

react_prompt = hub.pull("hwchase17/react")  # template de ReAct

tavily_search = TavilySearchResults(max_results=3)
tools = [tavily_search]

agent = create_react_agent(llm, tools, react_prompt)  # inicializamos el agente ReAct
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)  # habilitamos ejecución de tools

/usr/local/lib/python3.10/dist-packages/langsmith/client.py:241: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


In [ ]:
# Realizamos una consulta de prueba
query = "¿Dónde habita la perdiz chilena?"
results = tavily_search.run(query)
print(results)

[{'url': 'https://conociendonuestrasaves.blogspot.com/2019/09/perdiz-chilena.html', 'content': 'Habitat: La perdiz chilena se encuentra en valles y matorrales de altura en altitudes desde los 400 a los 2000 msnm. También habita en trigales y bosques áridos, en asociación con árboles como el espino, Porlieria chilensis, y la palmera chilena. Paises en donde vive la/el Perdiz chilena'}, {'url': 'https://www.rutaschile.com/Guia-Aves-Detalle.php?N=Perdiz-chilena', 'content': 'Habitat: La perdiz chilena se encuentra en valles y matorrales de altura en altitudes desde los 400 a los 2000 msnm. También habita en trigales y bosques áridos, en asociación con árboles como el espino, Porlieria chilensis, y la palmera chilena. Paises en donde vive la/el Perdiz chilena CHILE'}, {'url': 'https://animalia.bio/es/chilean-tinamou', 'content': 'La perdiz chilena, inambú chileno o yuto cordillerano (Nothoprocta perdicaria), es un ave de la familia de los tinámidos.Al igual que otras especies de esta famil

#### **2.2.2 Tool de Wikipedia (0.2 puntos)**

Generar una *tool* que pueda hacer consultas a **Wikipedia**.

*Hint: Le puede ser de ayuda el siguiente [link](https://python.langchain.com/v0.1/docs/modules/tools/).*

In [ ]:
!pip install langchain wikipedia-api

  Preparing metadata (setup.py) ... done
  Created wheel for wikipedia-api: filename=Wikipedia_API-0.7.1-py3-none-any.whl size=14346 sha256=8f702304fc9120c5966b50a480dbf3510fcb7c58692f8d26eb552bb43f1d80f8
  Stored in directory: /root/.cache/pip/wheels/4c/96/18/b9201cc3e8b47b02b510460210cfd832ccf10c0c4dd0522962
Successfully built wikipedia-api


In [ ]:
!pip install wikipedia

  Preparing metadata (setup.py) ... done
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11679 sha256=09574fc30ffec6d0245d1c864a3e40e3ac4918a730f2291a704e8748f9d0d787
  Stored in directory: /root/.cache/pip/wheels/5e/b6/c5/93f3dec388ae76edc830cb42901bb0232504dfc0df02fc50de
Successfully built wikipedia


In [ ]:
# Generar una tool que pueda hacer consultas a Wikipedia.
from langchain.tools import WikipediaQueryRun
from langchain.utilities import WikipediaAPIWrapper

wikipedia_api_wrapper = WikipediaAPIWrapper(
    lang="es",
    top_k_results=3
)

wikipedia_search = WikipediaQueryRun(api_wrapper=wikipedia_api_wrapper)
tools = [wikipedia_search]

In [ ]:
# Consulta de prueba:
query = "¿Dónde habita la perdiz chilena?"
result = wikipedia_search.run(query)
print(result)

Page: Gastronomía de Chile
Summary: La gastronomía de Chile es producto de la mezcla entre la tradición indígena y el aporte colonial español,[1]​ combinando sus alimentos, costumbres y hábitos culinarios.[2]​[3]​ A lo largo del tiempo, ha tenido aportes menores de cocinas europeas por parte de inmigrantes, como la alemana e italiana; sin embargo, en el siglo XX tuvo una importante y marcada influencia de la cocina francesa.[1]​ Estos elementos conformaron lo que se conoce como «cocina criolla chilena», la cual destaca por sus variados sabores, ingredientes y colores,[4]​ resultado de la diversidad geográfica del país,[3]​[5]​ acompañada de bebidas alcohólicas como el pisco y el vino chilenos.
Los platos más tradicionales de la cocina chilena son el ajiaco, los anticuchos, los asados, la calapurca, el cancato, la carbonada, la cazuela, el chapalele, el charquicán, el curanto, las empanadas de pino, las humitas, el milcao, la paila marina, la pantruca, el pastel de choclo,[6]​ el pastel

#### **2.2.3 Crear Agente (0.3 puntos)**

Crear un agente que pueda responder preguntas preguntas usando las *tools* antes generadas. Asegúrese que su agente responda en español. Por último, guarde el agente en una variable.

In [ ]:
# Crear un agente que pueda responder preguntas preguntas usando las tools antes generadas. Asegúrese que su agente responda en español. Por último, guarde el agente en una variable.
from langchain.agents import initialize_agent, Tool
#from langchain.chat_models import ChatGoogleGenerativeAI
from langchain.prompts import PromptTemplate

from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain.prompts import PromptTemplate

# Utilizaremos mismo modelo LLM de antes

tools = [tavily_search, wikipedia_search]

prompt = PromptTemplate.from_template(
    """
    Eres un agente experto en responder preguntas en español usando herramientas externas.
    Usa las herramientas disponibles como Tavily y Wikipedia para buscar información relevante.
    Siempre utiliza las herramientas antes de proporcionar una respuesta.

    Pregunta del usuario: {input}
    Pasos a seguir: {agent_scratchpad}
    Respuesta basada en los datos proporcionados:
    """
)

agent = create_tool_calling_agent(llm, tools, prompt)

agent_tools = AgentExecutor(agent=agent, tools=tools, verbose=True)


#### **2.2.4 Verificación de respuestas (0.3 puntos)**

Pruebe el funcionamiento de su agente y asegúrese que el agente esté ocupando correctamente las tools disponibles. ¿En qué casos el agente debería ocupar la tool de Tavily? ¿En qué casos debería ocupar la tool de Wikipedia?

In [ ]:
question = "¿Dónde habita la perdiz chilena?"
response = agent_tools.invoke({"input": question})

print("Respuesta del agente:")
print(response)



> Entering new AgentExecutor chain...

Invoking: `wikipedia` with `{'query': 'perdiz chilena'}`


Page: Nothoprocta perdicaria
Summary: La perdiz chilena (Nothoprocta perdicaria),[2]​ también llamado tinamú chileno[3]​ o inambú chileno,[4]​ es un ave de la familia de los tinámidos. Al igual que otras especies de esta familia, es conocida vulgarmente con el nombre de perdiz dada su similitud muy superficial con la perdiz europea (Perdix perdix). Es endémica de la zona central de Chile, habitando pastizales semiáridos, campos agrícolas y claros, así como matorrales y bosques en la zona del matorral chileno,[5]​ además de haber sido introducida a la isla de Pascua.[6]​

Page: Nothoprocta perdicaria perdicaria
Summary: La perdiz chilena del norte o inambú chileno del norte  (Nothoprocta perdicaria perdicaria) es la subespecie nortina de un ave de la familia de los tinámidos: la Perdiz chilena o inambú chileno. Al igual que otras especies de esta familia, es conocida vulgarmente con el no

<p align="left">
  <img src="https://i.pinimg.com/1200x/fd/d0/32/fdd0329acbf93a0b7df286dc2557e87a.jpg"
" width="200">
</p>

### **2.3 Multi Agente (1.5 puntos)**

<p align="center">
  <img src="https://media1.tenor.com/m/r7QMJLxU4BoAAAAd/this-is-getting-out-of-hand-star-wars.gif"
" width="450">
</p>

El objetivo de esta subsección es encapsular las funcionalidades creadas en una solución multiagente con un **supervisor**.


#### **2.3.1 Generando Tools (0.5 puntos)**

Transforme la solución RAG de la sección 2.1 y el agente de la sección 2.2 a *tools* (una tool por cada uno).

In [ ]:
!pip install langchain

In [ ]:
# Transformación RAG de la sección 2.1
from langchain.agents import Tool

fauna_tool = Tool(
    name="Fauna Tool",
    func=lambda input: str(rag_chain.invoke({"query": input})),
    description="Usa la herramienta Fauna Tool para responder preguntas relacionadas con la fauna chilena"
)

# Transformación Agent sección 2.2
agent_tool = Tool(
    name="Agent Tool",
    func=lambda input: agent_tools.invoke({"input":input}),
    description="Usa la herramienta Agent Tool para responder cualquier pregunta usando Tavily y Wikipedia."
)

#### **2.3.2 Agente Supervisor (0.5 puntos)**

Habilite un agente que tenga acceso a las tools del punto anterior y pueda responder preguntas relacionadas. Almacene este agente en una variable llamada supervisor.

In [ ]:
from langchain.agents import initialize_agent

tools = [fauna_tool, agent_tool]

supervisor_prompt = PromptTemplate.from_template(
    """
    Eres un agente supervisor experto. Tienes acceso a las siguientes herramientas:
    - Fauna Tool: Para responder preguntas relacionadas con la fauna chilena.
    - Agent Tool: Para realiza búsquedas generales usando Tavily y Wikipedia.

    Analiza la pregunta del usuario y decide qué herramienta usar.
    Sólo utiliza una herramienta por pregunta.

    Pregunta: {input}
    Respuesta:
    """
)

supervisor = initialize_agent(
    tools=tools,
    llm=llm,
    agent="zero-shot-react-description",
    verbose=True,
    agent_kwargs={"prompt": supervisor_prompt}
)

#### **2.3.3 Verificación de respuestas (0.25 puntos)**

Pruebe el funcionamiento de su agente repitiendo las preguntas realizadas en las secciones 2.1.4 y 2.2.4 y comente sus resultados. ¿Cómo varían las respuestas bajo este enfoque?

In [ ]:
question = "¿Dónde habita la perdiz chilena?"
response = supervisor.run(question)
print(response)



> Entering new AgentExecutor chain...
Thought: Necesito información sobre la ubicación geográfica de la perdiz chilena.  La herramienta Fauna Tool parece la más adecuada para esto.

Action: Fauna Tool
Action Input: ¿Dónde habita la perdiz chilena?

Observation: La perdiz chilena habita en campos de pastizales, arbustos bajos y campos agrícolas.

Thought:Thought: La respuesta de Fauna Tool es correcta pero incompleta.  Necesitaría información más específica sobre su distribución geográfica en Chile.  Usaré Agent Tool para complementar la información.

Action: Agent Tool
Action Input: ¿Cuál es la distribución geográfica de la perdiz chilena en Chile?


> Entering new AgentExecutor chain...

Invoking: `wikipedia` with `{'query': 'Distribución geográfica de la perdiz chilena en Chile'}`
responded: Primero, usaré la herramienta `wikipedia` para buscar información sobre la distribución geográfica de la perdiz chilena en Chile.



Page: Nothoprocta perdicaria
Summary: La perdiz chilena (Not

<p align="left">
  <img src="https://i.pinimg.com/1200x/fd/d0/32/fdd0329acbf93a0b7df286dc2557e87a.jpg"
" width="200">
</p>

#### **2.3.4 Análisis (0.25 puntos)**

¿Qué diferencias tiene este enfoque con la solución *Router* vista en clases? Nombre al menos una ventaja y desventaja.

El **Agente Superviso**r realizado es un único modelo, mediante el promp definido, analiza la entrada del usuario y luego decide cuál tool utilizar. Donde la decición la tiene exclusivamente el modelo LLM. El `Router` visto en clases, utiliza una lógica externa (cómo reglas definidas) para definir qué tool usar antes de realizar la consulta.

* Una **ventaja** del **Agente Supervisor** es que puede adaptarse fácilmente a preguntas más complejas, debido a la capacidad que tiene de razonamiento lógico.

* Una **desventaja** del **Agente Supervisor** es que si el modelo LLM no logra clasificar bien la pregunta que está realizando el usuario, puede malinterpretarla, eligiendo una tool incorrecta.

### **2.4 Memoria (Bonus +0.5 puntos)**

<p align="center">
  <img src="https://media1.tenor.com/m/Gs95aiElrscAAAAd/memory-unlocked-ratatouille-critic.gif"
" width="400">
</p>

Una de las principales falencias de las soluciones que hemos visto hasta ahora es que nuestro chat no responde las interacciones anteriores, por ejemplo:

- Pregunta 1: "Hola! mi nombre es Sebastián"
  - Respuesta esperada: "Hola Sebastián! ..."
- Pregunta 2: "Cual es mi nombre?"
  - Respuesta actual: "Lo siento pero no conozco tu nombre :("
  - **Respuesta esperada: "Tu nombre es Sebastián"**

Para solucionar esto, se les solicita agregar un componente de **memoria** a la solución entregada en el punto 2.3.

**Nota: El Bonus es válido <u>sólo para la sección 2 de Large Language Models.</u>**

In [ ]:
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationChain

memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True
)

supervisor_with_memory = initialize_agent(
    tools=[fauna_tool, agent_tool],
    llm=llm,
    agent="conversational-react-description",
    verbose=True,
    memory=memory  # agregar memoria al agente
)

In [ ]:
# Primera interacción
response_1 = supervisor_with_memory.run("Hola! ¿Cómo se llama nuestro grupo?")
print("Respuesta 1:", response_1)




> Entering new AgentExecutor chain...
```tool_code
Thought: Do I need to use a tool? No
AI: Hola!  No tengo información sobre el nombre de nuestro grupo, ya que no tenemos un historial de conversación previo.  ¿Puedes contarme más?
```


> Finished chain.
Respuesta 1: Hola!  No tengo información sobre el nombre de nuestro grupo, ya que no tenemos un historial de conversación previo.  ¿Puedes contarme más?
```


In [ ]:
response_2 = supervisor_with_memory.run("Nuestro grupo se llama VA & RP.")
print("Respuesta 2:", response_2)



> Entering new AgentExecutor chain...
```tool_code
Thought: Do I need to use a tool? No
AI: ¡Ah, genial!  Ahora entiendo.  ¿En qué puedo ayudar a VA & RP?
```


> Finished chain.
Respuesta 2: ¡Ah, genial!  Ahora entiendo.  ¿En qué puedo ayudar a VA & RP?
```


In [ ]:
response_3 = supervisor_with_memory.run("¿Cómo se llama nuestro grupo?")
print("Respuesta 3:", response_3)



> Entering new AgentExecutor chain...
```tool_code
Thought: Do I need to use a tool? No
AI: Nuestro grupo se llama VA & RP.
```


> Finished chain.
Respuesta 3: Nuestro grupo se llama VA & RP.
```


### **2.5 Despliegue (0 puntos)**

<p align="center">
  <img src="https://media1.tenor.com/m/IytHqOp52EsAAAAd/you-get-a-deploy-deploy.gif"
" width="400">
</p>

Una vez tengan los puntos anteriores finalizados, toca la etapa de dar a conocer lo que hicimos! Para eso, vamos a desplegar nuestro modelo a través de `gradio`, una librería especializada en el levantamiento rápido de demos basadas en ML.

Primero instalamos la librería:

In [ ]:
%pip install --upgrade --quiet gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.1/57.1 MB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.1/320.1 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 60.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.2/73.2 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.8/63.8 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 8.3 MB/s eta 0:00:00


Luego sólo deben ejecutar el siguiente código e interactuar con la interfaz a través del notebook o del link generado:

In [ ]:
agent_response = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [ ]:
import gradio as gr
import time

def agent_response(message, history):
  '''
  Función para gradio, recibe mensaje e historial, devuelte la respuesta del chatbot.
  '''
  # get chatbot response
  response = ... # rellenar con la respuesta de su chat

  # assert
  assert type(response) == str, "output de route_question debe ser string"

  # "streaming" response
  for i in range(len(response)):
    time.sleep(0.015)
    yield response[: i+1]

gr.ChatInterface(
    agent_response,
    type="messages",
    title="👻Chatbot VA & RP👻", # Pueden cambiar esto si lo desean
    description="Hola! Soy un chatbot fantasma 👻", # también la descripción
    theme="soft",
    ).launch(
        share=True, # pueden compartir el link a sus amig@s para que interactuen con su chat!
        debug = False,
        )

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c543c9b5f815528877.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
